# Episode 6: Anatomy of an Agent

You've built agents with a name, a prompt, and a config. But the `Agent` constructor has more slots — and together they form a composable **harness**.

**In this episode you'll learn:** every slot in the `Agent` constructor, and which episode goes deep on each.

This is the opening episode of **Group 2 — Agent Harness**. The next three episodes each take one slot in detail.

## The Agent is a harness

A bare agent is just a prompt plus a model. A *production* agent also needs to log, retry, stay within a token budget, return typed data, remember things, and stay safe.

In the Beta API you don't subclass or wrap the agent to get those. You **plug capabilities into constructor slots**. The agent is a harness; each slot is independent; you opt into what you need.

## The slots

Every parameter of the `Agent` constructor, and where this workshop covers it:

| Slot | What it does | Covered in |
|---|---|---|
| `name` | Identifies the agent | Episode 2 |
| `prompt` | The system prompt — role and behavior | Episode 2 |
| `config` | Which model and provider | Episode 2 |
| `tools` | Functions and sub-agents the model can call | Episodes 3–4 |
| `middleware` | Intercepts turns, LLM calls, and tool calls | **Episode 7** |
| `assembly` | Shapes what enters the prompt each turn | **Episode 8** |
| `response_schema` | Returns a typed object instead of text | **Episode 9** |
| `observers` | Watch the event stream and raise alerts | Episode 25 |
| `knowledge` | Persistent memory / RAG | Episode 19 |
| `tasks` | Auto-injected sub-task delegation tools | Episode 4 |

You've already used `name`, `prompt`, `config`, `tools`, and `tasks`. This group covers `middleware`, `assembly`, and `response_schema`.

## Setup

The usual setup — load the API key and build a model config.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig
from autogen.beta.middleware import LoggingMiddleware

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)

## One agent, several slots

Here's an agent wiring three slots at once: `config`, `tools`, and `middleware`. Each is independent — adding middleware doesn't change the tool, and removing the tool doesn't change the middleware.

`LoggingMiddleware` is a built-in (Episode 7 covers middleware in full); it quietly logs the turn through Python's logging system.

In [1]:
def now_utc() -> str:
    """Return the current time (a fixed value for this demo)."""
    return "2026-05-19 14:30 UTC"


assistant = Agent(
    "assistant",
    prompt="You are a concise assistant. Use tools when relevant.",
    config=config,  # slot: which model
    tools=[now_utc],  # slot: what it can do
    middleware=[LoggingMiddleware()],  # slot: cross-cutting behavior
)

reply = await assistant.ask("What time is it? Answer in one sentence.")
print(reply.body)

The current time is 14:30 UTC on May 19, 2026.


## What just happened

That one `ask()` flowed through the whole harness:

1. **Middleware** wrapped the turn (logging start/end).
2. The model decided to call the **tool** `now_utc`.
3. The tool result went back to the model, which wrote the final answer.

You changed the agent's capabilities purely by what you passed to the constructor — no subclassing, no wrappers.

## Additional content

**Slots also work per-call.** `agent.ask(...)` and `reply.ask(...)` accept `tools=`, `middleware=`, `config=`, and `response_schema=` too. Constructor slots apply to *every* turn; call-level slots apply to *one* turn and are appended after the constructor's.

**Independence is the point.** Because slots don't know about each other, you can test an agent with no middleware, then add `LoggingMiddleware` for debugging, then swap in `TelemetryMiddleware` for production — without touching the prompt, tools, or model.

**The harness scales down too.** A one-line `Agent(name, prompt=..., config=...)` is the same class. You pay only for the slots you fill.

## What's next

Next we go deep on the first harness slot: **middleware** — how to intercept and shape every turn, LLM call, and tool execution.